# **Donor-Level Data Split**
---

This notebook creates train/validation/test splits with **P7 as the test donor**. The split column must be added to the `adata.obs` of the previously preprocessed .h5ad file prior to tokenization, so that the split labels are preserved in the tokenized dataset's metadata. This allows the Geneformer `prepare_data` function to correctly split the data for training/validation/testing.

## Why Donor-Level Splitting?

- Models can learn donor-specific batch effects
- Testing on held-out donors ensures true generalization
- P7 is completely held out from training

## Importance for Cell Type Classification Fine-Tuning

When fine-tuning Geneformer for cell type classification:

**Problem with random splits:**
- Same donors appear in train and test sets (data leakage)
- Model learns donor-specific technical artifacts (batch effects, sequencing depth, donor genetics)
- High test accuracy may reflect **memorization of donor characteristics** rather than biological cell type features
- Overly optimistic performance estimates

**Solution with donor-level splits:**
- Test set contains cells from unseen donors (P7)
- Model **cannot memorize donor-specific patterns** for test cells
- Forces model to learn **generalizable cell type features** (gene expression signatures)
- Realistic performance estimate for deployment on new patient/donor samples

**Clinical relevance:**
- In real applications, models will classify cells from new patients
- Donor-level validation mirrors real-world usage
- Essential for assessing if model learns biology vs. batch effects
---

## Setup Instructions

Before running this notebook, update the paths in the configuration cell below:
- `BASE_DIR`: Root directory for the benchmarking repository
- `PREPROCESSED_DIR`: Directory containing preprocessed h5ad files
- `SPLIT_INDICES_DIR`: Output directory for split indices
- `PROCESSED_FILE`: Name of the preprocessed h5ad file (from notebook 01)

In [1]:
import pandas as pd
import scanpy as sc
import numpy as np
import json
import os
from pathlib import Path
from sklearn.model_selection import train_test_split

# ===== CONFIGURATION - Update these paths for your environment =====
BASE_DIR = Path("/scratch/tmurugan/geneformer_benchmarking")
PREPROCESSED_DIR = BASE_DIR / "data" / "preprocessed_data"
SPLIT_INDICES_DIR = BASE_DIR / "data" / "split_indices"

# Input/output files
PROCESSED_FILE = "citeseq_pbmc_geneformer_processed.h5ad"
SPLIT_NAME = "p7_test_donor_split"

# Construct paths
preprocessed_path = PREPROCESSED_DIR / PROCESSED_FILE
output_dir = SPLIT_INDICES_DIR / SPLIT_NAME

# Create output directory if it doesn't exist
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Configuration loaded:")
print(f"Base directory: {BASE_DIR}")
print(f"Preprocessed data: {preprocessed_path}")
print(f"Split indices output: {output_dir}")

Configuration loaded:
Base directory: /scratch/tmurugan/geneformer_benchmarking
Preprocessed data: /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/citeseq_pbmc_geneformer_processed.h5ad
Split indices output: /scratch/tmurugan/geneformer_benchmarking/data/split_indices/p7_test_donor_split


## Load Data and Check Donors

In [2]:
# Load the preprocessed dataset
adata = sc.read_h5ad(preprocessed_path)
print(adata)

# Check available donors
print(f"\nAvailable donors: {adata.obs['donor_id'].unique()}")
print(f"Number of donors: {adata.obs['donor_id'].nunique()}")
print(f"\nCells per donor:")
print(adata.obs['donor_id'].value_counts())

AnnData object with n_obs × n_vars = 161764 × 20264
    obs: 'nCount_ADT', 'nFeature_ADT', 'nCount_RNA', 'nFeature_RNA', 'orig.ident', 'lane', 'donor_id', 'time', 'celltype.l1', 'celltype.l2', 'celltype.l3', 'Phase', 'nCount_SCT', 'nFeature_SCT', 'cell_type_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'n_counts', 'joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'ensembl_id'
    uns: 'celltype.l3_colors', 'citation', 'default_embedding', 'neighbors', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_apca', 'X_aumap', 'X_pca', 'X_

## Create Split with P7 as Test Donor

**Strategy:**
1. Assign all P7 cells to test set
2. Split remaining cells into train/validation (stratified by cell type)

In [3]:
# Split parameters
test_donor = "P7"  # Donor to assign to test set
donor_key = "donor_id"
label_key = "celltype.l3"
split_column = label_key.split('.')[-1] + "_" + "split"   # e.g., "l3_split"

# Assign P7 to test set
adata.obs[split_column] = "train"
test_mask = adata.obs[donor_key] == test_donor
adata.obs.loc[test_mask, split_column] = "test"

# Calculate test proportion
test_prop = test_mask.sum() / adata.n_obs
print(f"Test set ({test_donor}): {test_mask.sum()} cells ({test_prop:.1%})")

# Get remaining cells (non-test)
remaining_idx = adata.obs.index[~test_mask].to_numpy()
remaining_labels = adata.obs.loc[remaining_idx, label_key].astype(str)

# Split remaining into train/validation
# Validation = 0.15 of TOTAL dataset
# Convert to relative fraction of remaining data
val_frac_of_total = 0.15
remaining_total = 1.0 - test_prop
relative_val_frac = val_frac_of_total / remaining_total

train_idx, val_idx = train_test_split(
    remaining_idx,
    test_size=relative_val_frac,
    stratify=remaining_labels,
    random_state=42
)

# Assign validation
adata.obs.loc[val_idx, split_column] = "validation"

print(f"\nFinal split:")
print(adata.obs[split_column].value_counts())
print(f"\nProportions:")
print(adata.obs[split_column].value_counts(normalize=True))

Test set (P7): 25871 cells (16.0%)

Final split:
l3_split
train         111628
test           25871
validation     24265
Name: count, dtype: int64

Proportions:
l3_split
train         0.690067
test          0.159931
validation    0.150002
Name: proportion, dtype: float64


## Analyze Split Distribution

In [4]:
# Check distribution by cell type
per_label = adata.obs.groupby(label_key)[split_column].value_counts().unstack(fill_value=0)
print("Distribution by cell type:")
print(per_label)

# Check distribution by donor
per_donor = adata.obs.groupby(donor_key)[split_column].value_counts().unstack(fill_value=0)
print(f"\nDistribution by donor:")
print(per_donor)

Distribution by cell type:
l3_split               test  train  validation
celltype.l3                                   
ASDC_mDC                 12     23           5
ASDC_pDC                 10     21           5
B intermediate kappa    253    696         151
B intermediate lambda   414    753         164
B memory kappa          632   1154         251
B memory lambda         377    715         156
B naive kappa          1013   3154         685
B naive lambda          680   1796         390
CD4 CTL                 514   1004         218
CD4 Naive              2441  12353        2685
CD4 Proliferating        22     71          15
CD4 TCM_1               979   5883        1279
CD4 TCM_2               126    384          83
CD4 TCM_3               617   4549         989
CD4 TEM_1               144   1283         279
CD4 TEM_2                50    330          72
CD4 TEM_3               266   1461         317
CD4 TEM_4                30     41           9
CD8 Naive              1160   765

/tmp/ipykernel_4072528/4056706488.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_label = adata.obs.groupby(label_key)[split_column].value_counts().unstack(fill_value=0)
/tmp/ipykernel_4072528/4056706488.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  per_donor = adata.obs.groupby(donor_key)[split_column].value_counts().unstack(fill_value=0)


## Save Split Indices

In [5]:
# Get indices for each split
train_idx = np.where(adata.obs[split_column] == "train")[0]
val_idx = np.where(adata.obs[split_column] == "validation")[0]
test_idx = np.where(adata.obs[split_column] == "test")[0]

# Save to CSV
pd.Series(train_idx).to_csv(output_dir / "train_idx.csv", index=False, header=False)
pd.Series(val_idx).to_csv(output_dir / "val_idx.csv", index=False, header=False)
pd.Series(test_idx).to_csv(output_dir / "test_idx.csv", index=False, header=False)

print(f"Saved indices to: {output_dir}")
print(f"  Train: {len(train_idx)} cells")
print(f"  Validation: {len(val_idx)} cells")
print(f"  Test: {len(test_idx)} cells")

# Save provenance info
provenance = {
    "test_donor": test_donor,
    "split_counts": {
        "train": int(len(train_idx)),
        "validation": int(len(val_idx)),
        "test": int(len(test_idx))
    },
    "n_cells_total": int(adata.n_obs),
    "random_state": 42,
    "label_key": label_key
}

with open(output_dir / "split_provenance.json", "w") as f:
    json.dump(provenance, f, indent=2)

print(f"\nSaved provenance to: {output_dir / 'split_provenance.json'}")

Saved indices to: /scratch/tmurugan/geneformer_benchmarking/data/split_indices/p7_test_donor_split
  Train: 111628 cells
  Validation: 24265 cells
  Test: 25871 cells

Saved provenance to: /scratch/tmurugan/geneformer_benchmarking/data/split_indices/p7_test_donor_split/split_provenance.json


## Save AnnData with Split Column

In [6]:
# Save the dataset with split column
output_path = PREPROCESSED_DIR / SPLIT_NAME / f"citeseq_pbmc_with_{test_donor.lower()}_{split_column}.h5ad"

# Create parent directory if it doesn't exist
output_path.parent.mkdir(parents=True, exist_ok=True)

adata.write(output_path, compression="gzip")
print(f"Saved dataset with split column to: {output_path}")

Saved dataset with split column to: /scratch/tmurugan/geneformer_benchmarking/data/preprocessed_data/p7_test_donor_split/citeseq_pbmc_with_p7_l3_split.h5ad


---

## Summary

Created donor-level split with **P7 as test donor**
- All P7 cells → Test set
- Remaining cells → Train/Validation (stratified by cell type)
- Train/Val split maintains cell type proportions

**Files saved:**
- Split indices: `{output_dir}/train_idx.csv`, `val_idx.csv`, `test_idx.csv`
- Provenance: `{output_dir}/split_provenance.json`
- AnnData with split column: `citeseq_pbmc_with_{test_donor}_{split_column}.h5ad`